In [1]:
# 1. Cài đặt thư viện hệ thống cho DeepSpeed
!apt-get update && apt-get install -y libaio-dev

# 2. Xóa sạch tận gốc các thư viện cũ gây xung đột
!pip uninstall -y transformers trl peft accelerate openrlhf datasets
!rm -rf /usr/local/lib/python3.12/dist-packages/transformers*
!rm -rf /usr/local/lib/python3.12/dist-packages/peft*
!rm -rf /usr/local/lib/python3.12/dist-packages/trl*

# 3. Cài đặt lại với các phiên bản ổn định
!pip install "transformers>=4.57.6,<5.0.0" trl peft accelerate datasets bitsandbytes
!pip install git+https://github.com/OpenRLHF/OpenRLHF.git

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,785 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get

In [2]:
import json
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import GRPOTrainer, GRPOConfig

# ==========================================
# 1. CHUẨN BỊ DỮ LIỆU
# ==========================================
data_path = "/kaggle/input/datasets/moaaztameer/medqa-usmle/MedQA-USMLE/questions/US/train.jsonl"

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

raw_data = load_jsonl(data_path)

PROMPT_TEMPLATE = """A conversation between User and Assistant.
The assistant thinks internally and then answers.

The reasoning must be inside <think></think>
The final answer must be inside <answer></answer>

User:
{question}

Options:
{options}

Assistant:
"""

formatted_data = []
for example in raw_data:
    options_text = "\n".join([f"{k}. {v}" for k, v in example.get("options", {}).items()])
    prompt = PROMPT_TEMPLATE.format(question=example["question"], options=options_text)
    
    formatted_data.append({
        "prompt": prompt,
        "answer": example.get("answer_idx", example.get("answer", ""))  # Lấy đáp án đúng
    })

# Giới hạn 1000 mẫu để test chạy thử (bạn có thể bỏ slice [:1000] sau khi code chạy mượt)
train_dataset = Dataset.from_list(formatted_data[:500])

# ==========================================
# 2. LOAD MODEL 4-BIT & TOKENIZER
# ==========================================
model_name = "Qwen/Qwen2.5-3B"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Cấu hình nén 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",       # Chuẩn nén nf4 tối ưu nhất cho LLM
    bnb_4bit_compute_dtype=torch.bfloat16, 
    bnb_4bit_use_double_quant=True   # Nén kép để tiết kiệm VRAM thêm nữa
)

# Load model với cấu hình bnb_config
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,  # Đưa cấu hình 4-bit vào đây
    device_map="auto",
    trust_remote_code=True
)
# ==========================================
# 3. HÀM REWARD (PHẦN THƯỞNG CHO RLHF)
# ==========================================
def mcqa_reward_func(prompts, completions, answer, **kwargs):
    rewards = []
    # GRPOTrainer truyền danh sách các câu trả lời (completions) và đáp án (answer) tương ứng
    for response, gold in zip(completions, answer):
        # Trích xuất nội dung trong thẻ <answer> nếu có
        if "<answer>" not in response:
            rewards.append(-1.0)  # Phạt nặng nếu không tuân thủ format
        elif f"<answer> {gold} </answer>" in response or f"<answer>{gold}</answer>" in response:
            rewards.append(1.0)   # Thưởng nếu trả lời đúng
        else:
            rewards.append(0.0)   # Không thưởng không phạt nếu sai đáp án nhưng đúng format
    return rewards



config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [3]:
# ==========================================
# 4. CẤU HÌNH LORA & GRPO TRAINER
# ==========================================
# Bắt buộc dùng LoRA cho model 3B trên Kaggle T4 để không bị Out Of Memory (OOM)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

# Cấu hình GRPO
training_args = GRPOConfig(
    output_dir="./medqa-qwen-grpo",
    learning_rate=2e-5,
    per_device_train_batch_size=1, # Giữ ở mức 1 để tránh OOM
    gradient_accumulation_steps=2,
    #max_prompt_length=512,
    #max_completion_length=256,
    num_train_epochs=1,
    num_generations=2,             # GRPO cần sinh ít nhất 2 câu trả lời để so sánh reward
    logging_steps=5,
    report_to="none"               # Tắt wandb nếu bạn chưa setup
)

trainer = GRPOTrainer(
    model=base_model,
    reward_funcs=[mcqa_reward_func],
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
)


In [4]:
# ==========================================
# 5. BẮT ĐẦU TRAINING
# ==========================================
print("Bắt đầu huấn luyện...")
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Bắt đầu huấn luyện...


Passing `generation_config` together with generation-related arguments=({'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Step,Training Loss
5,0.110786
10,0.135201
15,0.058971
20,0.049410
25,0.046668
30,-0.078071
35,0.036918
40,0.008689
45,0.026965
50,0.088163


TrainOutput(global_step=500, training_loss=0.025493788212537765, metrics={'train_runtime': 37243.8934, 'train_samples_per_second': 0.013, 'train_steps_per_second': 0.013, 'total_flos': 0.0, 'train_loss': 0.025493788212537765})